# **Project Name** - **Voyage Analytics: Integrating MLOps in Travel**
##### **Project Type** - Regression, Classification, Recommendation + MLOps
##### **Contribution** - Individual
##### **Name** - Mayuri Laddha
##### **GitHub Repo** - https://github.com/Mayuriladdha/voyage-analytics-mlops  *(replace with your final repo link)*

# **Project Summary**
This capstone uses three linked travel datasets (users, flights, hotels) to build three models and then productionise the regression model end to end.

* **Regression** - predicts a flight's price from route, flight type, agency and travel date. Random Forest reached an RMSE of about 2.4 and an R² of about 0.9999 on a held-out 20% test set, against 103 RMSE for a linear baseline.
* **Classification** - predicts a user's gender (male / female / none). The data holds almost no signal for this task: every model lands near the 33% chance level, and the notebook explains why.
* **Recommendation** - suggests hotels with item-based collaborative filtering. It beats random but only matches a popularity baseline, because hotel choice looks almost uniform in this data.
* **MLOps** - the regression model is served with a Flask REST API, packaged in Docker, deployed on Kubernetes (with autoscaling), retrained on a schedule with Apache Airflow, delivered by a Jenkins CI/CD pipeline and tracked/versioned with MLflow. A Streamlit app shows the recommender and travel insights. All code is in the single GitHub repo above.

# **Problem Statement**
#### **Define Your Business Objective?**
A travel platform wants to (1) quote realistic flight prices instantly, (2) understand and segment its customers, and (3) suggest hotels that fit each traveller. A notebook alone cannot do this: the models must be served in real time, scaled with demand, retrained as new bookings arrive, and versioned so that every deployed model can be traced. The objective is to build the three models and a production-style MLOps system around the price model.

# ***Let's Begin !***
## ***1. Know Your Data***
### Import Libraries

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             accuracy_score, f1_score, classification_report, confusion_matrix)
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

### Dataset Loading

In [ ]:
# In Colab: upload users.csv, flights.csv and hotels.csv (from travel_capstone.zip) when asked.
if not os.path.exists("flights.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        pass   # running locally: put the CSVs next to the notebook

users   = pd.read_csv("users.csv")
flights = pd.read_csv("flights.csv")
hotels  = pd.read_csv("hotels.csv")

### Dataset First View

In [ ]:
for name, d in [("users", users), ("flights", flights), ("hotels", hotels)]:
    print(f"--- {name} ---")
    display(d.head()) if "display" in globals() else print(d.head())

### Dataset Rows & Columns count

In [ ]:
for name, d in [("users", users), ("flights", flights), ("hotels", hotels)]:
    print(f"{name}: {d.shape[0]:,} rows x {d.shape[1]} columns")

### Dataset Information

In [ ]:
for d in (users, flights, hotels):
    d.info(); print()

#### Duplicate Values

In [ ]:
for name, d in [("users", users), ("flights", flights), ("hotels", hotels)]:
    print(name, "duplicate rows:", d.duplicated().sum())

#### Missing Values/Null Values

In [ ]:
for name, d in [("users", users), ("flights", flights), ("hotels", hotels)]:
    print(name, "missing values:", int(d.isna().sum().sum()))

### What did you know about your dataset?
* **users** has 1,340 travellers from 5 companies with a gender label (male / female / none, roughly balanced) and age 21-65.
* **flights** has about 272 thousand flights between 9 Brazilian cities, with price, duration, distance, flight type (economic / premium / firstClass) and agency (Rainbow / CloudFy / FlyingDrops).
* **hotels** has about 40.5 thousand stays in 9 hotels (one hotel per city), with stay length (1-4 days), price per day and total.
* There are no missing values. The hotels file contains a single duplicate row, which is dropped in wrangling.
* `userCode` links flights and hotels to users (`users.code`), and `travelCode` links a flight to the hotel stay of the same trip.

## ***2. Understanding Your Variables***

In [ ]:
print(users.describe(include="all").T, "\n")
print(flights.describe(include="all").T, "\n")
print(hotels.describe(include="all").T)

### Variables Description
**users:** `code` user id | `company` employer | `name` | `gender` | `age`
**flights:** `travelCode` trip id | `userCode` user id | `from`, `to` cities | `flightType` cabin class | `price` | `time` duration (hours) | `distance` (km) | `agency` | `date`
**hotels:** `travelCode` trip id | `userCode` user id | `name` hotel | `place` city | `days` nights | `price` per night | `total` = days x price | `date`

### Check Unique Values for each variable.

In [ ]:
for name, d in [("users", users), ("flights", flights), ("hotels", hotels)]:
    print(f"--- {name} ---")
    print(d.nunique(), "\n")

## 3. ***Data Wrangling***
### Data Wrangling Code

In [ ]:
hotels = hotels.drop_duplicates()
flights["date"] = pd.to_datetime(flights["date"], format="%m/%d/%Y")
hotels["date"]  = pd.to_datetime(hotels["date"],  format="%m/%d/%Y")
flights["year"], flights["month"], flights["dayofweek"] = flights.date.dt.year, flights.date.dt.month, flights.date.dt.dayofweek

# checks that justify later modelling choices
print("time vs distance correlation :", round(flights["time"].corr(flights["distance"]), 5))
print("distances per route (max)    :", flights.groupby(["from", "to"])["distance"].nunique().max())
print("hotel total == days*price    :", np.allclose(hotels["total"], hotels["days"] * hotels["price"]))

# one merged table for analysis: flights + user profile
flights_users = flights.merge(users.rename(columns={"code": "userCode"}), on="userCode", how="left")
print(flights_users.shape)

### What all manipulations have you done and insights you found?
* Removed the duplicate hotel row and converted both date columns to datetime, then derived `year`, `month` and `dayofweek`.
* `time` and `distance` have a correlation of 0.99999, so they carry the same information. Only `distance` is used as a model input, which avoids redundant, collinear features.
* Each (from, to) route has exactly one distance, so the deployed API only needs the route: it looks the distance up itself and the caller never has to supply it.
* `total` is exactly `days x price`, so it would leak the target for any hotel price model and is not used as a feature.
* Merged flights with users to analyse price by traveller attributes.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1 - Distribution of flight price (Univariate)

In [ ]:
plt.figure(figsize=(8,4)); sns.histplot(flights["price"], bins=60, kde=True)
plt.title("Flight price distribution"); plt.xlabel("price"); plt.show()

##### 1. Why did you pick the specific chart?
A histogram shows the shape and spread of the numeric target before any modelling.

##### 2. What is/are the insight(s) found from the chart?
Prices range from about 300 to 1,750 and average about 957. The distribution is multi-modal, which suggests that discrete factors such as cabin class drive price.

##### 3. Will the gained insights help creating a positive business impact?
Yes. Knowing the price range helps the platform set sensible limits and alerts, and the multi-modal shape tells us to look at flight type next.

#### Chart - 2 - Flight type vs price (Bivariate: categorical vs numeric)

In [ ]:
plt.figure(figsize=(8,4)); sns.boxplot(data=flights, x="flightType", y="price", order=["economic","premium","firstClass"])
plt.title("Price by flight type"); plt.show()
print(flights.groupby("flightType")["price"].mean().round(1))

##### 1. Why did you pick the specific chart?
A box plot compares the price distribution of each category side by side and shows spread and outliers.

##### 2. What is/are the insight(s) found from the chart?
Average price is about 658 for economic, 920 for premium and 1,181 for firstClass, and the three boxes barely overlap. Flight type is the strongest categorical driver of price.

##### 3. Will the gained insights help creating a positive business impact?
Yes. It confirms that flight type must be a model input and supports tiered pricing and upsell offers (economic to premium).

#### Chart - 3 - Distance vs price (Bivariate: numeric vs numeric)

In [ ]:
s = flights.sample(8000, random_state=1)
plt.figure(figsize=(8,5)); sns.scatterplot(data=s, x="distance", y="price", hue="flightType", alpha=.5, s=14)
plt.title("Price vs distance"); plt.show()
print("corr(price, distance) =", round(flights["price"].corr(flights["distance"]), 3))

##### 1. Why did you pick the specific chart?
A scatter plot coloured by flight type shows the numeric relationship and whether it differs by group.

##### 2. What is/are the insight(s) found from the chart?
Price rises with distance (correlation about 0.64), and each flight type forms its own band. Distance alone is not enough: cabin class shifts the whole price level.

##### 3. Will the gained insights help creating a positive business impact?
Yes. A model needs distance and flight type together. It also shows how much extra a customer pays per km in each class.

#### Chart - 4 - Agency vs flight type (Bivariate: categorical vs categorical)

In [ ]:
plt.figure(figsize=(8,4)); sns.countplot(data=flights, x="agency", hue="flightType"); plt.title("Flights by agency and type"); plt.show()
print(flights.groupby(["agency","flightType"])["price"].mean().round(0))

##### 1. Why did you pick the specific chart?
A grouped count plot shows how the categories combine, so it can reveal structural quirks in the data.

##### 2. What is/are the insight(s) found from the chart?
FlyingDrops sells only firstClass tickets (about 38.8 thousand), while CloudFy and Rainbow sell all three classes in similar numbers. FlyingDrops looks about 29% more expensive on average only because of this mix: within firstClass, all three agencies cost about the same (1,177-1,186).

##### 3. Will the gained insights help creating a positive business impact?
Yes. The agency is not a price driver by itself, so the business should not read FlyingDrops as 'premium pricing'. It is a first-class-only carrier, which is useful for partnership and inventory decisions.

#### Chart - 5 - Average price by route (Bivariate)

In [ ]:
r = flights.groupby(["from","to"])["price"].mean().sort_values()
r = pd.concat([r.head(5), r.tail(5)])
plt.figure(figsize=(8,5)); r.plot(kind="barh"); plt.title("5 cheapest and 5 most expensive routes (avg price)"); plt.show()

##### 1. Why did you pick the specific chart?
A horizontal bar chart ranks many categories clearly and lets long city names stay readable.

##### 2. What is/are the insight(s) found from the chart?
The cheapest routes average about 475-481 (Florianopolis to Rio, Brasilia to Rio) and the most expensive about 1,371-1,381 (Campo Grande to Rio, Sao Paulo to Florianopolis), so route has a large effect on price.

##### 3. Will the gained insights help creating a positive business impact?
Yes. Route-level price benchmarks support route planning, promotions on cheap routes and capacity decisions on expensive ones.

#### Chart - 6 - Flights per year and average price per month (Time series)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
flights.groupby("year").size().plot(kind="bar", ax=ax[0], title="Flights per year")
flights.groupby("month")["price"].mean().plot(marker="o", ax=ax[1], title="Average price by month", ylim=(900,1000))
plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?
Line and bar charts over time show volume trends and seasonality.

##### 2. What is/are the insight(s) found from the chart?
Volumes peak in 2020 (about 112.6 thousand flights) and fall every year after to about 6.4 thousand in 2023, which is partly because the 2023 data is incomplete. Monthly average price is almost flat (about 951-963), so there is no real seasonality in price.

##### 3. Will the gained insights help creating a positive business impact?
Yes. Price does not vary by season, so the model can use month as a minor feature. The volume decline is a warning sign for demand and data coverage that the business should investigate, and a reason to retrain the model regularly.

#### Chart - 7 - Hotel price per night and stays by city (Bivariate)

In [ ]:
g = hotels.groupby("place").agg(avg_price=("price","mean"), stays=("days","size")).sort_values("avg_price")
fig, ax = plt.subplots(1, 2, figsize=(13,4))
g["avg_price"].plot(kind="barh", ax=ax[0], title="Average hotel price per night")
g["stays"].plot(kind="barh", ax=ax[1], title="Number of stays")
plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?
Bar charts compare a numeric measure across the nine hotel cities.

##### 2. What is/are the insight(s) found from the chart?
Price per night varies a lot by hotel: Campo Grande's Hotel BW is cheapest (about 60), Sao Paulo's Hotel AF about 139, and Florianopolis' Hotel A and Recife's Hotel AU the most expensive (about 313). Stays are spread fairly evenly, between about 3.3 and 5.1 thousand per city.

##### 3. Will the gained insights help creating a positive business impact?
Yes. It shows where price-sensitive customers can be offered budget options and where premium stays dominate revenue.

#### Chart - 8 - Users: age and gender (Univariate + Bivariate)

In [ ]:
plt.figure(figsize=(8,4)); sns.kdeplot(data=users, x="age", hue="gender", common_norm=False, fill=True, alpha=.25)
plt.title("Age distribution by gender"); plt.show()
print(users.groupby("gender")["age"].agg(["count","mean"]).round(1))

##### 1. Why did you pick the specific chart?
A KDE plot compares the age distribution of each gender group in one view.

##### 2. What is/are the insight(s) found from the chart?
Genders are balanced (452 male, 448 female, 440 none) and the age distributions overlap almost completely (mean age about 42-43 for all three).

##### 3. Will the gained insights help creating a positive business impact?
This is a warning for the classification task: if the groups look alike, a model has little to learn from. It warns us before modelling that gender will be hard to predict.

#### Chart - 9 - Correlation Heatmap

In [ ]:
plt.figure(figsize=(7,5)); sns.heatmap(flights[["price","time","distance","year","month","dayofweek"]].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heatmap (flights)"); plt.show()

##### 1. Why did you pick the specific chart?
A heatmap summarises linear relationships between all numeric variables in one picture.

##### 2. What is/are the insight(s) found from the chart?
`time` and `distance` are almost perfectly correlated (0.99999) and both correlate about 0.64 with price. `month`, `dayofweek` and `year` have almost no correlation with price.

##### 3. Will the gained insights help creating a positive business impact?
Yes. It justifies dropping `time` and keeping the model compact, and it shows that calendar fields add little signal.

## ***5. Regression - Flight Price Prediction***
### Feature selection and train/test split

**Features:** `from`, `to`, `flightType`, `agency` (categorical, one-hot encoded) and `distance`, `month`, `dayofweek` (numeric). `time` is dropped (duplicate of distance), `year` is dropped (a model should not extrapolate on a calendar year), and identifiers (`travelCode`, `userCode`) are dropped because they carry no pricing logic. Data is split 80/20 with a fixed seed. The preprocessing sits inside a scikit-learn `Pipeline`, so exactly the same transformation runs in training and inside the Flask API.

In [ ]:
CAT = ["from","to","flightType","agency"]; NUM = ["distance","month","dayofweek"]
X, y = flights[CAT+NUM], flights["price"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_tr.shape, X_te.shape)

### Model training and comparison

In [ ]:
def pipe(model):
    prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)], remainder="passthrough")
    return Pipeline([("prep", prep), ("model", model)])

models = {
    "Linear Regression (baseline)": LinearRegression(),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=60, max_depth=16, min_samples_leaf=3, n_jobs=-1, random_state=42),
}
rows, fitted = [], {}
for name, m in models.items():
    p = pipe(m).fit(X_tr, y_tr); pred = p.predict(X_te); fitted[name] = p
    rows.append({"model": name, "RMSE": mean_squared_error(y_te, pred)**.5, "MAE": mean_absolute_error(y_te, pred), "R2": r2_score(y_te, pred)})
results = pd.DataFrame(rows).set_index("model").round(4)
results

### Evaluation and interpretation

In [ ]:
best = fitted["Random Forest"]; pred = best.predict(X_te)
fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].scatter(y_te[:3000], pred[:3000], s=6, alpha=.4); ax[0].plot([200,1800],[200,1800],"r--"); ax[0].set_title("Actual vs predicted"); ax[0].set_xlabel("actual"); ax[0].set_ylabel("predicted")
sns.histplot(y_te - pred, bins=60, ax=ax[1]); ax[1].set_title("Residuals"); plt.tight_layout(); plt.show()

names = best.named_steps["prep"].get_feature_names_out()
imp = pd.Series(best.named_steps["model"].feature_importances_, index=names).sort_values(ascending=False).head(8)
imp.plot(kind="barh", title="Top 8 feature importances (Random Forest)").invert_yaxis(); plt.show()

**Interpretation.**
* The linear model explains about 92% of the variance (RMSE about 103) but it cannot capture the interaction of flight type, route and distance. Hist Gradient Boosting cuts RMSE to about 16. Random Forest is the best, with an RMSE of about 2.4, a MAE of about 0.23 and R² of about 0.9999, and it is the model that is deployed.
* The features that matter are `distance` and the flight-type indicators, which matches the EDA (Charts 2, 3 and 5). Calendar fields and the agency contribute almost nothing.
* **Caution:** an R² this close to 1 means that prices in this dataset are almost a deterministic function of route and cabin class (it looks synthetic). Real fares depend on booking lead time, demand and competitors, so the same accuracy should not be expected on live data. That is exactly why the MLOps part (monitoring, scheduled retraining and model versioning) matters.

## ***6. Classification - Gender Prediction***

Users are described by `age` and `company`, plus behaviour aggregated per user from the other two files: number of flights, average flight price and distance, share of first-class flights, number of hotel stays, average hotel total and average stay length. The target has three balanced classes, so macro-F1 is reported next to accuracy, and a majority-class baseline is included to show what "no learning" looks like.

In [ ]:
fa = flights.groupby("userCode").agg(n_flights=("travelCode","size"), avg_flight_price=("price","mean"), avg_distance=("distance","mean"),
                                     share_first_class=("flightType", lambda s: (s=="firstClass").mean()))
ha = hotels.groupby("userCode").agg(n_hotel_stays=("travelCode","size"), avg_hotel_total=("total","mean"), avg_stay_days=("days","mean"))
ut = users.rename(columns={"code":"userCode"}).merge(fa, on="userCode", how="left").merge(ha, on="userCode", how="left").fillna(0)

UNUM = ["age","n_flights","avg_flight_price","avg_distance","share_first_class","n_hotel_stays","avg_hotel_total","avg_stay_days"]
Xc, yc = ut[UNUM+["company"]], ut["gender"]
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, stratify=yc, random_state=42)
prep = ColumnTransformer([("num", StandardScaler(), UNUM), ("cat", OneHotEncoder(handle_unknown="ignore"), ["company"])])
clfs = {"Majority baseline": DummyClassifier(strategy="most_frequent"),
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=42)}
crow, cfit = [], {}
for n, m in clfs.items():
    p = Pipeline([("prep", prep), ("model", m)]).fit(Xc_tr, yc_tr); cfit[n] = p; pr = p.predict(Xc_te)
    crow.append({"model": n, "accuracy": accuracy_score(yc_te, pr), "macro_F1": f1_score(yc_te, pr, average="macro")})
pd.DataFrame(crow).set_index("model").round(3)

In [ ]:
print(classification_report(yc_te, cfit["Logistic Regression"].predict(Xc_te)))
sns.heatmap(confusion_matrix(yc_te, cfit["Logistic Regression"].predict(Xc_te), labels=["female","male","none"]), annot=True, fmt="d",
            xticklabels=["female","male","none"], yticklabels=["female","male","none"]); plt.title("Confusion matrix"); plt.show()

**Interpretation.** Accuracy is about 0.32-0.34 for every model, the same as the 0.34 chance level for three balanced classes, and macro-F1 is about 0.32-0.33. The confusion matrix is spread evenly across cells. Age and travel behaviour simply do not separate the gender classes in this dataset (Chart 8 showed identical age profiles), so the honest conclusion is that **this model should not be used to make real decisions**. It is deployed in the API (`/predict_gender`) to demonstrate the serving pattern, and the result also shows why model monitoring matters. Better results would need new, genuinely informative features, and predicting a person's gender from behaviour should also be reviewed for fairness and privacy before any real use.

## ***7. Recommendation - Hotel Suggestions***

**Approach: item-based collaborative filtering.** A user x hotel matrix is built from the nights each user spent in each hotel (implicit feedback). Cosine similarity between hotel columns says which hotels are visited by the same kind of users. A user's score for an unvisited hotel is the sum of similarities to the hotels they already visited, weighted by nights. Cold-start users (unknown ids) get the most popular hotels.

In [ ]:
def fit_reco(h):
    m = h.pivot_table(index="userCode", columns="name", values="days", aggfunc="sum", fill_value=0)
    sim = pd.DataFrame(cosine_similarity(m.T), index=m.columns, columns=m.columns)
    info = h.groupby("name").agg(place=("place","first"), avg_price_per_day=("price","mean"), stays=("travelCode","size"))
    return {"m": m, "sim": sim, "info": info, "pop": info["stays"].sort_values(ascending=False)}

def recommend(model, user, k=3):
    m, sim, pop = model["m"], model["sim"], model["pop"]
    if user not in m.index: names = list(pop.index[:k])
    else:
        seen = m.loc[user]; scores = sim.mul(seen, axis=0).sum(axis=0)
        names = list(scores[seen == 0].sort_values(ascending=False).index[:k])
        names += [n for n in pop.index if n not in names][:k-len(names)]
    return model["info"].loc[names].round(1)

reco = fit_reco(hotels)
recommend(reco, 0, 3)

**Evaluation (leave-last-hotel-out, hit-rate@3).** Most users have already visited 7-9 of the 9 hotels, which makes "predict a hotel they have not seen" trivial. The test therefore uses only users with 3-5 distinct hotels. For each of them the most recently first-visited hotel is hidden, and we check whether it appears in the top 3. Two baselines are shown: random among unseen hotels and the most popular unseen hotels.

In [ ]:
hs = hotels.sort_values("date")
fv = hs.groupby(["userCode","name"])["date"].min().reset_index()
hit_cf = hit_pop = rnd = n = 0
for uid, g in fv.groupby("userCode"):
    if not 3 <= len(g) <= 5: continue
    held = g.sort_values("date").iloc[-1]["name"]
    mdl = fit_reco(hs[~((hs.userCode == uid) & (hs.name == held))])
    seen = set(mdl["m"].loc[uid][mdl["m"].loc[uid] > 0].index)
    unseen = [x for x in mdl["pop"].index if x not in seen]
    hit_cf += held in list(recommend(mdl, uid, 3).index); hit_pop += held in unseen[:3]; rnd += min(3, len(unseen))/len(unseen); n += 1
    if n >= 300: break
pd.Series({"users evaluated": n, "CF hit@3": hit_cf/n, "popularity hit@3": hit_pop/n, "random hit@3": rnd/n}).round(3)

**Interpretation.** Collaborative filtering (about 0.61) beats random (about 0.53) but is no better than simply recommending the most popular unseen hotels (about 0.63) on this sample. The reason is that hotel choice here looks nearly uniform: no strong taste patterns exist for the model to learn. The recommender still works as a safe default with a cold-start fallback, and the Streamlit app shows it with each user's history. With richer signals (ratings, price sensitivity, trip purpose, destination of the booked flight), content-based or hybrid models would be the next step.

## ***8. Conclusion***

* **Regression:** a Random Forest pipeline predicts flight price with RMSE about 2.4 (R² about 0.9999) and is the model deployed through Flask, Docker, Kubernetes, Airflow, Jenkins and MLflow. The near-perfect score reflects the structure of this dataset, so live monitoring and retraining are essential.
* **Classification:** performance is at chance level (about 33%). The honest result is that the available features do not predict gender, and the model is not suitable for real decisions.
* **Recommendation:** item-based CF works and handles cold-start users, but it only matches a popularity baseline here.
* **MLOps:** the same preprocessing + model pipeline is shared by training, API and app. Scheduled retraining (Airflow) with an RMSE quality gate, versioned runs (MLflow) and automated build-and-deploy (Jenkins to Kubernetes) turn a notebook into a repeatable system.
* **Next steps:** model monitoring and drift alerts, authentication on the API, richer features for classification and recommendation, and a cloud deployment.

### ***Congrats! Your model and notebook are complete!***